# 🏥 Med-Bench-Arena — Colab runner for medical & TCM LLMs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pariskang/Med-Bench-Arena/blob/main/notebooks/Med_Bench_Arena_Colab.ipynb)

Pick **one** of 18 medical / TCM LLMs and score it across **all** wired benchmarks
(MCQ · open-ended · safety · 中医辨证/方剂 · agents). This notebook drives
[`Med-Bench-Arena`](https://github.com/pariskang/Med-Bench-Arena): it clones the repo,
installs vLLM, runs the model against every dataset it can reach, and **saves all results
to your Google Drive** so they persist after the session ends.

> **GPU tiers** (`Runtime → Change runtime type`):
> - **T4 (free, 16GB)** → ≤7B: `zhongjing-2-1_8b`, `biancang`, `biomistral-7b`,
>   `huatuogpt-o1-7b`, `clinicalgpt-r1`, `taiyi`, `lingshu-7b`, `huatuogpt2-7b`, `aquilamed-rl`.
> - **A100 (40GB)** → 13–34B: `disc-medllm`, `baichuan-m1-14b`, `dao1-30b-a3b`,
>   `deepseek-r1-32b`, `baichuan-m2-32b`, `medgemma-27b-it`.
> - **2×A100 / H100 80GB** → 70B: `meditron-70b`, `citrus-70b`.
>
> One model per session — vLLM keeps the model resident in GPU memory.

## 1 · Check the GPU

In [ ]:
!nvidia-smi

## 2 · Install Med-Bench-Arena + vLLM

Takes a few minutes (vLLM + torch). Restart the runtime only if Colab asks.

In [ ]:
# Clone the repo (use your fork/branch if needed)
![ -d Med-Bench-Arena ] || git clone https://github.com/pariskang/Med-Bench-Arena.git
%cd Med-Bench-Arena

!pip install -q -e .
# datasets = benchmark loading · litellm = optional real judge · peft = ZhongJing LoRA
!pip install -q 'datasets>=2.18' litellm peft nest_asyncio
# vLLM pulls a compatible torch/transformers (the Dao series uses transformers).
!pip install -q vllm
print('\n✅ install done — if Colab shows a "restart" banner, restart then re-run from here.')

## 3 · Mount Google Drive (results are saved here)

Every leaderboard + per-sample detail file is written under `RESULTS_ROOT` on your Drive,
so the run survives the session and you can resume it later (generations are cached there).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RESULTS_ROOT = '/content/drive/MyDrive/Med-Bench-Arena/results'  #@param {type:'string'}
import os; os.makedirs(RESULTS_ROOT, exist_ok=True)
print('results will be saved under:', RESULTS_ROOT)

## 4 · (Optional) HuggingFace login — only for GATED models

`meditron-70b` and `medgemma-27b-it` are gated: accept the license on the HF model page,
then paste a token (https://huggingface.co/settings/tokens). Skip for every other model.

In [ ]:
from huggingface_hub import notebook_login
# notebook_login()   # uncomment for gated models (Meditron / MedGemma)

## 5 · Pick a model + choose the benchmark scope

`MODEL_ID` must be one of the catalog ids (printed below). **`LIMIT = 0` (the default) scores
*every* question in *every* dataset — the full benchmark.** Set a small number (e.g. `50`) only
for a quick smoke test. `CATALOG_CONFIGS` decides which benchmark sets to sweep — by default every
text catalog. Each repo id / dtype / context was verified against the live HF page (see
[`MODELS.md`](https://github.com/pariskang/Med-Bench-Arena/blob/main/MODELS.md)).

In [ ]:
import yaml
MODEL_CATALOG = 'configs/catalog_med_models.yaml'
cfg = yaml.safe_load(open(MODEL_CATALOG))
print('Available HF model ids:')
for m in cfg['models']:
    if m.get('type') == 'hf': print('  -', m['id'])

In [ ]:
#@title Selection
MODEL_ID = 'zhongjing-2-1_8b'  #@param {type:'string'}
LIMIT    = 0                   #@param {type:'integer'}   # samples per dataset: 0 = ALL questions (full benchmark). Set e.g. 50 ONLY for a quick smoke test.
GPU_MEM_UTIL = 0.90            #@param {type:'number'}    # lower to 0.80 if you OOM on load

# Which benchmark catalogs to sweep (datasets are unioned + de-duplicated by id).
CATALOG_CONFIGS = [
    'configs/catalog_mcq.yaml',            # MedQA · MedMCQA · PubMedQA · MMLU · CMB · CMExam · TCMBench
    'configs/catalog_en_med.yaml',         # MedXpertQA · MedCalc · MedHallu · MLEC-QA · MediQ …
    'configs/catalog_cn_tcm.yaml',         # PromptCBLUE · TCMEval-PA · TCM-BEST4SDT · 名老中医医案
    'configs/catalog_ethics_safety.yaml',  # 医学伦理 · 安全红队 (CARES-18K …)
    'configs/example_tcm.yaml',            # 辨证 SDT · 方剂 · 安全 (judged)
    # 'configs/catalog_multimodal.yaml',   # uncomment ONLY for a vision model (lingshu / medgemma)
]

# Open-ended / safety / 辨证 datasets need an LLM judge. Default = offline mock judge
# (completes the run but its scores are placeholders). For REAL judge scores, set a key:
# import os; os.environ['DEEPSEEK_API_KEY'] = 'sk-...'   # uses deepseek-reasoner as judge

## 6 · Gather every dataset

Unions the `datasets:` blocks of the chosen catalogs (de-duped by id) into one sweep.

In [ ]:
seen, datasets = set(), []
for path in CATALOG_CONFIGS:
    try:
        c = yaml.safe_load(open(path))
    except FileNotFoundError:
        print('missing, skipped:', path); continue
    for d in c.get('datasets', []):
        if d['id'] in seen: continue
        seen.add(d['id']); datasets.append(d)
print(f'{len(datasets)} datasets queued:')
print('  ' + ', '.join(d['id'] for d in datasets))

## 7 · Run every benchmark → Google Drive

Loads the model **once**, then sweeps all datasets. `continue_on_error` keeps going when a
dataset is gated / offline / needs a server (it's skipped, not fatal). Generations are cached
under Drive, so re-running this cell resumes where it left off.

In [ ]:
import nest_asyncio; nest_asyncio.apply()   # Colab already runs an event loop
import os, copy, medeval

spec = next(m for m in cfg['models'] if m['id'] == MODEL_ID)
models = [copy.deepcopy(m) for m in cfg['models']
          if m['id'] == MODEL_ID or m.get('judge_only')]
for m in models:
    if m['id'] == MODEL_ID:
        m['gpu_memory_utilization'] = GPU_MEM_UTIL

# Real judge if a key is present, else the offline mock judge.
judge_id = 'mock-judge'
if os.environ.get('DEEPSEEK_API_KEY'):
    models.append({'id': 'deepseek-r1', 'type': 'litellm',
                   'model': 'deepseek/deepseek-reasoner',
                   'api_key_env': 'DEEPSEEK_API_KEY', 'judge_only': True})
    judge_id = 'deepseek-r1'

# Apply the sample budget per dataset. LIMIT = 0 -> FULL run: score EVERY question and
# drop any cap a catalog might carry. LIMIT > 0 caps each dataset (quick smoke test).
ds_specs = copy.deepcopy(datasets)
for d in ds_specs:
    if LIMIT:
        d['limit'] = LIMIT
    else:
        d.pop('limit', None)
scope = f'{LIMIT} samples/dataset' if LIMIT else 'ALL questions (full benchmark)'
print('judge:', judge_id, '| model:', MODEL_ID, '->', spec['model'])
print('scope:', scope, '| datasets:', len(ds_specs))
if not LIMIT:
    print('⚠️ full run — this can take hours; progress bars show per-sample status, results flush to Drive every 32 samples, re-running this cell resumes where it left off.')

run_cfg = {
    'run': {'output_dir': f'{RESULTS_ROOT}/{MODEL_ID}',
            'cache': True, 'concurrency': 16, 'continue_on_error': True, 'checkpoint_every': 32},
    'eval': {'gen': cfg.get('eval', {}).get('gen', {'temperature': 0.0, 'max_tokens': 2048}),
             'judge_model': judge_id},
    'models': models,
    'datasets': ds_specs,
}
rows = medeval.run_config(run_cfg)
print(f'\n✅ done — {len(rows)} (model × dataset) results in {run_cfg["run"]["output_dir"]}')

## 8 · Results (saved in your Drive)

In [ ]:
from pathlib import Path
OUT = Path(f'{RESULTS_ROOT}/{MODEL_ID}')
print(Path(OUT, 'leaderboard.md').read_text())

In [ ]:
# Per-dataset headline scores as a table
import json, pandas as pd
rows = json.loads(Path(OUT, 'leaderboard.json').read_text())
def headline(metrics):
    for _m, d in metrics.items():
        for k in ('accuracy','judge_score','pass^k','chain_score','herb_f1'):
            if k in d: return k, round(d[k], 4)
    return '', None
tbl = [{'dataset': r['dataset'], 'n': r['n'], 'split': r.get('split_type','official'),
        'metric': headline(r['metrics'])[0], 'score': headline(r['metrics'])[1]} for r in rows]
pd.DataFrame(tbl).sort_values('dataset')

In [ ]:
# Peek at a few graded samples from one dataset
import glob
for f in sorted(glob.glob(f'{OUT}/detail__*.jsonl'))[:1]:
    print('===', Path(f).name, '===')
    for line in open(f).readlines()[:3]:
        r = json.loads(line)
        print('Q   :', r['prompt'][:140].replace(chr(10),' '))
        print('pred:', str(r['parsed'])[:80], '| gold:', r['reference'])
        print('score:', {k: v['value'] for k, v in r['scores'].items()}, '\n')

## 9 · Notes

- **Sample budget (full vs. smoke)**: `LIMIT = 0` (the default) scores **every** question in every
  dataset — the full benchmark (tens of thousands of MCQ items: CMB-test alone is 11,200, CMExam
  ~6.8k, MedMCQA-val ~4.2k …), which can take **hours** on a single GPU. Set `LIMIT = 50` (or any N)
  for a fast smoke test. Either way generations cache to Drive, so re-running cell 7 **resumes**
  instead of regenerating — a long full run can survive a session restart.
- **Where results live**: `MyDrive/Med-Bench-Arena/results/<model_id>/` — `leaderboard.md`/`.json`,
  per-dataset `detail__*.jsonl`, and a `cache/` folder (so re-runs resume instead of regenerating).
- **Real judge**: open-ended / safety / 辨证 scores use the offline **mock** judge unless you set
  `os.environ['DEEPSEEK_API_KEY']` before cell 7 — then DeepSeek-R1 grades them. MCQ / numeric /
  structured-TCM metrics are always real (no judge needed).
- **Skipped datasets**: anything gated (needs HF terms), offline-only, or needing a live server
  (e.g. MedAgentBench's FHIR) is reported as skipped at the end of cell 7 — not a crash.
- **Dao1-30B-A3B** is pre-configured to dodge OOM: transformers backend + `device_map=auto`
  (offloads to CPU/disk), `float16` + `eager` attention, with Dao's sampling + 小道/Tao system
  prompt — all from its catalog entry. Runs even on a small GPU (slower); flip to `backend: vllm`
  on a big GPU.
- **Big models on small GPUs**: 30–70B won't fit a single T4/A100-40GB at bf16 — use a bigger
  runtime, set `tensor_parallel` to the GPU count, or point the id at a 4-bit/AWQ/GPTQ community repo.
- **Multimodal** (`lingshu-7b`, `medgemma-27b-it`): add `configs/catalog_multimodal.yaml` to
  `CATALOG_CONFIGS`; they run as text backends for the text MCQ sets here.